## Settings

In [ ]:
from dataclasses import dataclass
import torch


@dataclass
class AConfig:
    meta_file: str = "../processed/meta.json"

    image_model: str = "tf_efficientnetv2_s.in21k_ft_in1k"
    text_model: str = "vinai/phobert-base-v2"

    image_size: int = 300
    model_dim: int = 512
    heads: int = 8
    layers: int = 2

    question_len: int = 64
    answer_len: int = 32

    batch_size: int = 32
    workers: int = 0
    epochs: int = 25
    patience: int = 5

    encoder_lr: float = 2e-5
    task_lr: float = 5e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    grad_accum: int = 1
    grad_clip: float = 1.0

    fusion_dropout: float = 0.2
    decoder_dropout: float = 0.2
    label_smooth: float = 0.00
    text_layers: int = 4

    use_text_aug: bool = True
    text_aug_prob: float = 0.25

    seed: int = 42
    fp16: bool = True
    resume: bool = True
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

## Data tools

In [ ]:
import gc
import torch

class MemoryCleaner:
    @staticmethod
    def clear() -> None:
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

In [ ]:
import json
from pathlib import Path


class MetaRead:
    @staticmethod
    def load(path: str) -> dict[str, str]:
        with Path(path).open("r", encoding="utf-8") as file:
            data = json.load(file)

        return {str(key): str(value) for key, value in data.items()}

In [ ]:
import json
from pathlib import Path

class JsonRead:
    @staticmethod
    def list_data(path: str) -> list[dict[str, object]]:
        with Path(path).open("r", encoding="utf-8") as file:
            data = json.load(file)

        if not isinstance(data, list):
            raise ValueError(f"Expected list JSON: {path}")

        return data

In [ ]:
import random


class QuestionAugment:
    def __init__(self, enabled: bool, prob: float) -> None:
        self.enabled = enabled
        self.prob = prob
        self.rules = [
            ("là gì", "là gì vậy"),
            ("có bao nhiêu", "số lượng là bao nhiêu"),
            ("màu gì", "có màu gì"),
            ("ở đâu", "nằm ở đâu"),
            ("có phải", "có đúng là")
        ]

    def __call__(self, text: str) -> str:
        if not self.enabled:
            return text

        if random.random() > self.prob:
            return text

        new_text = text

        for old, new in self.rules:
            lower_text = new_text.lower()

            if old in lower_text:
                new_text = lower_text.replace(old, new)
                break

        return new_text

In [ ]:
from PIL import Image
from torch.utils.data import Dataset


class VQADataset(Dataset):
    def __init__(
        self,
        json_path: str,
        transform: object,
        text_aug: object | None = None
    ) -> None:
        self.items = JsonRead.list_data(json_path)
        self.transform = transform
        self.text_aug = text_aug

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, index: int) -> dict[str, object]:
        item = self.items[index]
        image = Image.open(str(item["image_path"])).convert("RGB")
        image = self.transform(image)
        question = str(item["question"])

        if self.text_aug is not None:
            question = self.text_aug(question)

        return {
            "image": image,
            "question": question,
            "answer": str(item["answer"])
        }

In [ ]:
from torchvision import transforms


class ImageTransform:
    @staticmethod
    def train(size: int) -> transforms.Compose:
        return transforms.Compose(
            [
                transforms.RandomResizedCrop(
                    size,
                    scale=(0.88, 1.0),
                    ratio=(0.95, 1.05)
                ),
                transforms.RandomRotation(5),
                transforms.ColorJitter(
                    brightness=0.12,
                    contrast=0.12,
                    saturation=0.10,
                    hue=0.01
                ),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)
                )
            ]
        )

    @staticmethod
    def eval(size: int) -> transforms.Compose:
        return transforms.Compose(
            [
                transforms.Resize((size, size)),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)
                )
            ]
        )

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

class BatchCollator:
    def __init__(self, tokenizer: object, config: AConfig) -> None:
        self.tokenizer = tokenizer
        self.config = config

    def __call__(self, batch: list[dict[str, object]]) -> dict[str, object]:
        questions = [str(item["question"]) for item in batch]
        answers = [str(item["answer"]) for item in batch]
        images = torch.stack([item["image"] for item in batch])

        q_data = self.tokenizer(
            questions,
            padding=True,
            truncation=True,
            max_length=self.config.question_len,
            return_tensors="pt"
        )
        a_data = self.tokenizer(
            answers,
            padding="max_length",
            truncation=True,
            max_length=self.config.answer_len,
            return_tensors="pt"
        )

        return {
            "images": images,
            "question_ids": q_data["input_ids"],
            "question_mask": q_data["attention_mask"],
            "answer_ids": a_data["input_ids"],
            "answers": answers
        }

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

class DataModule:
    def __init__(self, config: AConfig) -> None:
        self.config = config
        self.meta = MetaRead.load(config.meta_file)
        self.tokenizer = AutoTokenizer.from_pretrained(
            config.text_model,
            use_fast=False
        )

    def sets(self) -> tuple[VQADataset, VQADataset, VQADataset]:
        train_aug = QuestionAugment(
            enabled=self.config.use_text_aug,
            prob=self.config.text_aug_prob
        )
        train = VQADataset(
            self.meta["train_json"],
            ImageTransform.train(self.config.image_size),
            train_aug
        )
        val = VQADataset(
            self.meta["val_json"],
            ImageTransform.eval(self.config.image_size)
        )
        test = VQADataset(
            self.meta["test_json"],
            ImageTransform.eval(self.config.image_size)
        )

        return train, val, test

    def loader(self, data_set: VQADataset, shuffle: bool) -> DataLoader:
        return DataLoader(
            data_set,
            batch_size=self.config.batch_size,
            shuffle=shuffle,
            num_workers=self.config.workers,
            collate_fn=BatchCollator(self.tokenizer, self.config),
            pin_memory=self.config.device == "cuda"
        )

    def loaders(self) -> tuple[DataLoader, DataLoader, DataLoader]:
        train, val, test = self.sets()

        return (
            self.loader(train, True),
            self.loader(val, False),
            self.loader(test, False)
        )

## Model

In [ ]:
import timm
import torch
from torch import nn



class ImageEncoder(nn.Module):
    def __init__(self, name: str, model_dim: int) -> None:
        super().__init__()

        self.backbone = timm.create_model(
            name,
            pretrained=True,
            features_only=True,
            out_indices=(-1,)
        )
        channels = self.backbone.feature_info.channels()[-1]
        self.proj = nn.Conv2d(channels, model_dim, kernel_size=1)
        self.norm = nn.LayerNorm(model_dim)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        features = self.backbone(images)[-1]
        features = self.proj(features)
        features = features.flatten(2).transpose(1, 2)

        return self.norm(features)

In [ ]:
import torch
from torch import nn
from transformers import AutoModel


class TextEncoder(nn.Module):
    def __init__(
        self,
        name: str,
        model_dim: int,
        train_layers: int
    ) -> None:
        super().__init__()

        self.backbone = AutoModel.from_pretrained(
            name,
            add_pooling_layer=False
        )
        self.proj = nn.Linear(self.backbone.config.hidden_size, model_dim)
        self.norm = nn.LayerNorm(model_dim)
        self.unfreeze_tail(train_layers)

    def unfreeze_tail(self, train_layers: int) -> None:
        for param in self.backbone.parameters():
            param.requires_grad = False

        encoder = getattr(self.backbone, "encoder", None)
        layers = getattr(encoder, "layer", None)

        if layers is None:
            return

        for layer in layers[-train_layers:]:
            for param in layer.parameters():
                param.requires_grad = True

    def forward(
        self,
        question_ids: torch.Tensor,
        question_mask: torch.Tensor
    ) -> torch.Tensor:
        output = self.backbone(
            input_ids=question_ids,
            attention_mask=question_mask
        )
        hidden = self.proj(output.last_hidden_state)

        return self.norm(hidden)

In [ ]:
import torch
from torch import nn


class CoAttention(nn.Module):
    def __init__(self, model_dim: int, heads: int, dropout: float) -> None:
        super().__init__()

        self.text_to_image = nn.MultiheadAttention(
            embed_dim=model_dim,
            num_heads=heads,
            dropout=dropout,
            batch_first=True
        )
        self.image_to_text = nn.MultiheadAttention(
            embed_dim=model_dim,
            num_heads=heads,
            dropout=dropout,
            batch_first=True
        )
        self.text_norm = nn.LayerNorm(model_dim)
        self.image_norm = nn.LayerNorm(model_dim)
        self.dropout = nn.Dropout(dropout)

    def masked_mean(
        self,
        values: torch.Tensor,
        mask: torch.Tensor
    ) -> torch.Tensor:
        mask = mask.unsqueeze(-1).float()
        values = values * mask
        denom = mask.sum(dim=1).clamp_min(1.0)

        return values.sum(dim=1) / denom

    def forward(
        self,
        image_tokens: torch.Tensor,
        text_tokens: torch.Tensor,
        text_mask: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        text_mask = text_mask.bool()
        image_mask = torch.ones(
            image_tokens.size()[:2],
            dtype=torch.bool,
            device=image_tokens.device
        )
        text_attended, _ = self.text_to_image(
            query=text_tokens,
            key=image_tokens,
            value=image_tokens
        )
        image_attended, _ = self.image_to_text(
            query=image_tokens,
            key=text_tokens,
            value=text_tokens,
            key_padding_mask=~text_mask
        )
        text_tokens = self.text_norm(
            text_tokens + self.dropout(text_attended)
        )
        image_tokens = self.image_norm(
            image_tokens + self.dropout(image_attended)
        )
        memory = torch.cat([text_tokens, image_tokens], dim=1)
        memory_mask = torch.cat([text_mask, image_mask], dim=1)
        text_global = self.masked_mean(text_tokens, text_mask)
        image_global = self.masked_mean(image_tokens, image_mask)
        global_repr = 0.5 * (text_global + image_global)

        return memory, memory_mask, global_repr

In [ ]:
import math
import torch
from torch import nn


class LstmDecoder(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        model_dim: int,
        layers: int,
        pad_id: int,
        bos_id: int,
        eos_id: int,
        dropout: float
    ) -> None:
        super().__init__()

        self.pad_id = pad_id
        self.bos_id = bos_id
        self.eos_id = eos_id
        self.layers = layers
        self.model_dim = model_dim
        self.embedding = nn.Embedding(vocab_size, model_dim, pad_id)
        self.lstm = nn.LSTM(
            model_dim,
            model_dim,
            num_layers=layers,
            dropout=dropout if layers > 1 else 0.0,
            batch_first=True
        )
        self.h_proj = nn.Linear(model_dim, model_dim * layers)
        self.c_proj = nn.Linear(model_dim, model_dim * layers)
        self.join = nn.Linear(model_dim * 2, model_dim)
        self.output = nn.Linear(model_dim, vocab_size, bias=False)
        self.output.weight = self.embedding.weight

    def init_state(
        self,
        global_repr: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        batch = global_repr.size(0)
        h_state = self.h_proj(global_repr)
        c_state = self.c_proj(global_repr)
        h_state = h_state.view(batch, self.layers, self.model_dim)
        c_state = c_state.view(batch, self.layers, self.model_dim)

        return (
            h_state.transpose(0, 1).contiguous(),
            c_state.transpose(0, 1).contiguous()
        )

    def attend(
        self,
        query: torch.Tensor,
        memory: torch.Tensor,
        memory_mask: torch.Tensor
    ) -> torch.Tensor:
        scores = torch.bmm(query, memory.transpose(1, 2))
        scores = scores / math.sqrt(memory.size(-1))
        scores = scores.masked_fill(~memory_mask.unsqueeze(1), -1e4)
        weights = torch.softmax(scores, dim=-1)

        return torch.bmm(weights, memory)

    def forward(
        self,
        answer_ids: torch.Tensor,
        memory: torch.Tensor,
        memory_mask: torch.Tensor,
        global_repr: torch.Tensor
    ) -> torch.Tensor:
        inputs = answer_ids[:, :-1]
        hidden, _ = self.lstm(
            self.embedding(inputs),
            self.init_state(global_repr)
        )
        context = self.attend(hidden, memory, memory_mask)
        joined = self.join(torch.cat([hidden, context], dim=-1))

        return self.output(joined)

    def generate(
        self,
        memory: torch.Tensor,
        memory_mask: torch.Tensor,
        global_repr: torch.Tensor,
        max_len: int
    ) -> torch.Tensor:
        batch = memory.size(0)
        device = memory.device
        tokens = torch.full(
            (batch, 1),
            self.bos_id,
            dtype=torch.long,
            device=device
        )
        state = self.init_state(global_repr)
        finished = torch.zeros(batch, dtype=torch.bool, device=device)

        for _ in range(max_len - 1):
            hidden, state = self.lstm(self.embedding(tokens[:, -1:]), state)
            context = self.attend(hidden, memory, memory_mask)
            joined = self.join(torch.cat([hidden, context], dim=-1))
            logits = self.output(joined)
            next_token = logits[:, -1].argmax(dim=-1)
            next_token = next_token.masked_fill(finished, self.pad_id)
            tokens = torch.cat([tokens, next_token.unsqueeze(1)], dim=1)
            finished = finished | next_token.eq(self.eos_id)

            if bool(finished.all()):
                break

        return tokens

In [ ]:
import torch
from torch import nn


class TransformerDecoder(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        model_dim: int,
        heads: int,
        layers: int,
        answer_len: int,
        pad_id: int,
        bos_id: int,
        eos_id: int,
        dropout: float
    ) -> None:
        super().__init__()

        self.pad_id = pad_id
        self.bos_id = bos_id
        self.eos_id = eos_id
        self.embedding = nn.Embedding(vocab_size, model_dim, pad_id)
        self.position = nn.Embedding(answer_len, model_dim)
        layer = nn.TransformerDecoderLayer(
            d_model=model_dim,
            nhead=heads,
            dim_feedforward=model_dim * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.decoder = nn.TransformerDecoder(layer, num_layers=layers)
        self.norm = nn.LayerNorm(model_dim)
        self.output = nn.Linear(model_dim, vocab_size, bias=False)
        self.output.weight = self.embedding.weight

    def mask(self, size: int, device: torch.device) -> torch.Tensor:
        return torch.triu(
            torch.ones(size, size, dtype=torch.bool, device=device),
            diagonal=1
        )

    def forward(
        self,
        answer_ids: torch.Tensor,
        memory: torch.Tensor,
        memory_mask: torch.Tensor,
        global_repr: torch.Tensor
    ) -> torch.Tensor:
        inputs = answer_ids[:, :-1]
        length = inputs.size(1)
        pos = torch.arange(length, device=inputs.device).unsqueeze(0)
        hidden = self.embedding(inputs) + self.position(pos)
        hidden[:, 0] = hidden[:, 0] + global_repr
        output = self.decoder(
            tgt=hidden,
            memory=memory,
            tgt_mask=self.mask(length, inputs.device),
            tgt_key_padding_mask=inputs.eq(self.pad_id),
            memory_key_padding_mask=~memory_mask
        )

        return self.output(self.norm(output))

    def generate(
        self,
        memory: torch.Tensor,
        memory_mask: torch.Tensor,
        global_repr: torch.Tensor,
        max_len: int
    ) -> torch.Tensor:
        batch = memory.size(0)
        device = memory.device
        tokens = torch.full(
            (batch, 1),
            self.bos_id,
            dtype=torch.long,
            device=device
        )
        finished = torch.zeros(batch, dtype=torch.bool, device=device)

        for _ in range(max_len - 1):
            length = tokens.size(1)
            pos = torch.arange(length, device=device).unsqueeze(0)
            hidden = self.embedding(tokens) + self.position(pos)
            hidden[:, 0] = hidden[:, 0] + global_repr
            output = self.decoder(
                tgt=hidden,
                memory=memory,
                tgt_mask=self.mask(length, device),
                tgt_key_padding_mask=tokens.eq(self.pad_id),
                memory_key_padding_mask=~memory_mask
            )
            logits = self.output(self.norm(output))
            next_token = logits[:, -1].argmax(dim=-1)
            next_token = next_token.masked_fill(finished, self.pad_id)
            tokens = torch.cat([tokens, next_token.unsqueeze(1)], dim=1)
            finished = finished | next_token.eq(self.eos_id)

            if bool(finished.all()):
                break

        return tokens

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F


class VQAModel(nn.Module):
    def __init__(
        self,
        config: AConfig,
        decoder: str,
        vocab_size: int,
        pad_id: int,
        bos_id: int,
        eos_id: int
    ) -> None:
        super().__init__()

        self.config = config
        self.pad_id = pad_id
        self.image_encoder = ImageEncoder(
            config.image_model,
            config.model_dim
        )
        self.text_encoder = TextEncoder(
            config.text_model,
            config.model_dim,
            config.text_layers
        )
        self.fusion = CoAttention(
            config.model_dim,
            config.heads,
            config.fusion_dropout
        )

        if decoder == "lstm":
            self.decoder = LstmDecoder(
                vocab_size,
                config.model_dim,
                config.layers,
                pad_id,
                bos_id,
                eos_id,
                config.decoder_dropout
            )
        elif decoder == "transformer":
            self.decoder = TransformerDecoder(
                vocab_size,
                config.model_dim,
                config.heads,
                config.layers,
                config.answer_len,
                pad_id,
                bos_id,
                eos_id,
                config.decoder_dropout
            )
        else:
            raise ValueError(f"Invalid decoder: {decoder}")

    def encode(
        self,
        images: torch.Tensor,
        question_ids: torch.Tensor,
        question_mask: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        image_tokens = self.image_encoder(images)
        text_tokens = self.text_encoder(question_ids, question_mask)

        return self.fusion(image_tokens, text_tokens, question_mask)

    def forward(
        self,
        images: torch.Tensor,
        question_ids: torch.Tensor,
        question_mask: torch.Tensor,
        answer_ids: torch.Tensor
    ) -> dict[str, torch.Tensor]:
        memory, memory_mask, global_repr = self.encode(
            images,
            question_ids,
            question_mask
        )
        logits = self.decoder(answer_ids, memory, memory_mask, global_repr)
        targets = answer_ids[:, 1:]
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1),
            ignore_index=self.pad_id,
            label_smoothing=self.config.label_smooth
        )

        return {"loss": loss, "logits": logits}

    @torch.no_grad()
    def generate(
        self,
        images: torch.Tensor,
        question_ids: torch.Tensor,
        question_mask: torch.Tensor
    ) -> torch.Tensor:
        memory, memory_mask, global_repr = self.encode(
            images,
            question_ids,
            question_mask
        )

        return self.decoder.generate(
            memory,
            memory_mask,
            global_repr,
            self.config.answer_len
        )

## Evaluation helpers

In [ ]:
import numpy as np


class ExactMatchScore:
    def exact(self, pred: str, gold: str) -> float:
        pred = str(pred).lower().strip()
        gold = str(gold).lower().strip()

        return float(pred == gold)

    def batch(self, preds: list[str], golds: list[str]) -> dict[str, float]:
        scores = [
            self.exact(pred, gold)
            for pred, gold in zip(preds, golds)
        ]

        return {"exact_match": float(np.mean(scores))}

In [ ]:
from pathlib import Path
import torch

class Checkpoint:
    @staticmethod
    def load(path: Path, device: str) -> dict[str, object] | None:
        if not path.exists():
            return None

        return torch.load(path, map_location=device)

    @staticmethod
    def save(path: Path, data: dict[str, object]) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(data, path)

In [ ]:
class ModelBuild:
    def __init__(self, config: AConfig, tokenizer: object) -> None:
        self.config = config
        self.tokenizer = tokenizer

    def token_id(self, name: str) -> int:
        value = getattr(self.tokenizer, f"{name}_token_id", None)

        if value is not None:
            return int(value)

        if name == "bos":
            return int(self.tokenizer.cls_token_id)

        if name == "eos":
            return int(self.tokenizer.sep_token_id)

        return int(self.tokenizer.pad_token_id)

    def build(self, decoder: str) -> VQAModel:
        return VQAModel(
            config=self.config,
            decoder=decoder,
            vocab_size=len(self.tokenizer),
            pad_id=self.token_id("pad"),
            bos_id=self.token_id("bos"),
            eos_id=self.token_id("eos")
        )